In [1]:
import os
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, random_split
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image

/Users/griffin/Documents/ml_venv/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 1.4.23 (you have 1.4.22). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
class AugmentedECGDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels  # DataFrame containing labels and corresponding IDs
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Extract the image path and ID
        image_path = self.image_paths[idx]
        image_id = os.path.basename(image_path).split('_')[1].split('.')[0]  # Extract ID from path (e.g., '000100')

        # Find the corresponding row in the labels DataFrame using the ID
        label_row = self.labels[self.labels['ID'] == int(image_id)]  # Assuming 'ID' column is integers

        if label_row.empty:
            raise ValueError(f"ID {image_id} not found in the labels DataFrame.")

        # Extract the label values (assuming they are in the other columns of the DataFrame)
        labels = label_row.iloc[0, 1:].values  # Skipping the ID column

        # Try to load the image
        try:
            image = Image.open(image_path).convert("RGB")
        except (IOError, OSError) as e:
            print(f"Error loading image {image_path}: {e}")
            return None  # Skip this image if it cannot be loaded

        # Convert image to numpy array
        image = np.array(image)

        # Apply transformations (ensure the transformation works with named arguments)
        if self.transform:
            augmented = self.transform(image=image)  # Use 'image=image' to pass it as a named argument
            image = augmented['image']

        return {"pixel_values": image, "labels": torch.tensor(labels, dtype=torch.float32)}

In [3]:
# Load the CSV file with labels
labels_df = pd.read_csv("train_final.csv")

# We currently only have 1000 labels 
labels_df = labels_df[labels_df['ID'] < 1000]
labels_df = labels_df[labels_df['ID'] != 142] # corrupted file...

# Create a list of image paths based on the 'id' column
image_paths = [f"./data/train/train_{id_:06d}.png" for id_ in labels_df['ID']]

# Split the data into training and validation sets (80% train, 20% validation)
train_paths, val_paths, train_labels, val_labels = train_test_split(image_paths, labels_df, test_size=0.05, random_state=42)

In [4]:
# Define augmentations for training
train_transform = A.Compose([
    A.Resize(width=224, height=224),  # Resize to 224x224
    A.RandomRotate90(p=0.5),  # Randomly rotate by 90 degrees
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),  # Apply shift, scale, rotate
    A.ToGray(p=1.0),  # Convert to grayscale
    A.CLAHE(clip_limit=2.0, p=1.0),  # Contrast Limited Adaptive Histogram Equalization
    A.GaussianBlur(blur_limit=(3, 7), p=1.0),  # Apply Gaussian blur
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # Normalize
    ToTensorV2(),  # Convert to PyTorch tensor
])

In [5]:
# Define transformations for validation (no augmentation)
val_transform = A.Compose([
    A.Resize(width=224, height=224),
    A.ToGray(p=1.0),  # Convert to grayscale
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [6]:
# Create the datasets A training and validation
train_dataset = AugmentedECGDataset(train_paths, train_labels, transform=train_transform)
val_dataset = AugmentedECGDataset(val_paths, val_labels, transform=val_transform)

In [7]:
from torch.utils.data import DataLoader

batch_size = 5

# Create DataLoaders for training and validation datasets
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [8]:
from torchvision import models
import torch.nn as nn
import torch.optim as opt

# Load pretrained ResNet model
model = models.resnet18(pretrained=True)

# Modify the final layer for 5 output labels (multi-label classification)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 5)  # Output layer for 5 labels

criterion = nn.BCEWithLogitsLoss()

learning_rate = 0.01

optimizer = opt.Adam(
    model.parameters(),  # Parameters of the model you want to optimize
    lr=learning_rate,    # Learning rate
    betas=(0.9, 0.999),  # Beta values for the first and second moment estimates (default values are usually fine)
    eps=1e-8,            # A small constant added to improve numerical stability (default is fine)
    weight_decay=0       # L2 regularization (set to 0 to disable weight decay)
)

/Users/griffin/Documents/ml_venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/griffin/Documents/ml_venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
# Move model to the appropriate device (CUDA, MPS, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_built() else "cpu")

print(device)

# Move the model to the selected device
model = model.to(device)

mps


In [ ]:

num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    total_loss = 0.0

    for batch in train_dataloader:
        # Move batch data to GPU
        inputs = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {total_loss / len(train_dataloader)}")

    # Evaluate on validation data after every epoch
    model.eval()
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in val_dataloader:
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass
            outputs = model(inputs)

            # Apply sigmoid to get probabilities (since the output layer uses Sigmoid for multi-label classification)
            probabilities = torch.sigmoid(outputs)

            # Convert probabilities to binary predictions (threshold at 0.5)
            predictions = (probabilities > 0.5).float()

            # Count correct predictions
            correct_predictions += (predictions == labels).sum().item()
            total_predictions += labels.numel()  # Total number of labels in the batch

    # Calculate accuracy
    accuracy = correct_predictions / total_predictions
    print(f"Validation Accuracy: {accuracy:.4f}")

Epoch 1/10, Training Loss: 0.5431221788651065
Validation Accuracy: 0.7640
Epoch 2/10, Training Loss: 0.467007068662267
Validation Accuracy: 0.7840
